# Fine-tuning FinBERT on Nigerian financial headlines

**Run this on Kaggle** (Settings → Accelerator → GPU T4 ×2). The free tier gives
30 GPU-hours/week and supports background execution, which is the only GPU in
this project's stack. Everything else runs on GitHub Actions CPU runners.

## Why fine-tune at all

`ProsusAI/finbert` is trained on Financial PhraseBank — US/European corporate
news. Nigerian macro coverage uses vocabulary it has never seen weighted the way
we need: *naira*, *CBN*, *forex window*, *devaluation*, *subsidy removal*,
*parallel market*. The base model tends to read these as neutral.

## Output

Weights pushed to the HF Hub as `batestguy/finbert-ng-financial` (~440 MB — far
too large for git, and the Hub's free tier includes 100 GB). Switching the
pipeline over afterwards needs **no code change**: set the repo variable

```bash
gh variable set FINBERT_MODEL --body batestguy/finbert-ng-financial
```

`src/tobacco/nlp/finbert.py` reads that variable and `score.yml` passes it in.

In [ ]:
# scikit-learn is listed explicitly (train_test_split, below) rather than
# relying on Kaggle's preloaded image. It is deliberately NOT in the repo's
# requirements-actions.txt: no Actions job imports it, and this notebook never
# installs from that file.
!pip install -q transformers datasets accelerate evaluate huggingface_hub scikit-learn

## 1. Pull headlines from the repository

The scrapers have been committing headlines to `data/curated/news_articles/`
since the pipeline went live, so the repo *is* the training corpus. Read the
Parquet partitions straight off GitHub — public repo, no token needed.

In [ ]:
import io

import pandas as pd
import requests

REPO = "batestguy/tobacco-price-intelligence"
BRANCH = "main"

listing = requests.get(
    f"https://api.github.com/repos/{REPO}/contents/data/curated/news_articles",
    params={"ref": BRANCH},
    timeout=30,
).json()

frames = []
for entry in listing:
    if entry["name"].endswith(".parquet"):
        blob = requests.get(entry["download_url"], timeout=60).content
        frames.append(pd.read_parquet(io.BytesIO(blob)))

articles = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["id"])
print(f"{len(articles):,} headlines from {len(frames)} monthly partition(s)")
articles.head()

## 2. Labels

**This is the part that cannot be automated away.** The committed
`finbert_score` column is the *base model's* output; training on it would
distil the model's existing mistakes rather than correct them, and the
fine-tuned model would confidently reproduce exactly the errors we set out to
fix.

So: export a stratified sample for manual annotation. Sampling across the base
model's score range concentrates effort where it disagrees with itself, which
is where the label information actually is.

A few hundred labelled headlines is enough to move a domain-adapted classifier.
Label `0 = negative`, `1 = neutral`, `2 = positive`, judged as **impact on
Nigerian business conditions**, not on the sentiment of the sentence.

In [ ]:
scored = articles.dropna(subset=["finbert_score"]).copy()
scored["bucket"] = pd.cut(scored["finbert_score"], bins=5, labels=False)

sample = (
    scored.groupby("bucket", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 120), random_state=42))
    .loc[:, ["id", "headline", "url", "finbert_score"]]
)
sample["label"] = ""  # fill in: 0 negative, 1 neutral, 2 positive
sample.to_csv("headlines_to_label.csv", index=False)
print(f"Wrote {len(sample)} rows to headlines_to_label.csv — annotate, then re-upload.")

In [ ]:
# After annotating, upload the file as a Kaggle dataset and point this at it.
LABELLED_PATH = "/kaggle/input/ng-headlines-labelled/headlines_labelled.csv"

labelled = pd.read_csv(LABELLED_PATH)
labelled = labelled.dropna(subset=["label"])
labelled["label"] = labelled["label"].astype(int)
print(labelled["label"].value_counts().sort_index())
assert len(labelled) >= 200, "Too few labels to fine-tune meaningfully."

## 3. Fine-tune

The split is stratified and held out before any training so the reported
accuracy is measured on headlines the model has never seen.

In [ ]:
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

BASE_MODEL = "ProsusAI/finbert"

train_df, eval_df = train_test_split(
    labelled, test_size=0.2, stratify=labelled["label"], random_state=42
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def encode(batch):
    return tokenizer(batch["headline"], truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df[["headline", "label"]], preserve_index=False).map(encode, batched=True)
eval_ds = Dataset.from_pandas(eval_df[["headline", "label"]], preserve_index=False).map(encode, batched=True)

# Keep the base model's label order so downstream code that looks for a label
# starting with "neg" keeps working unchanged.
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2},
    ignore_mismatched_sizes=True,
)

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)
    return {
        **accuracy.compute(predictions=predictions, references=labels),
        **f1.compute(predictions=predictions, references=labels, average="macro"),
    }

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="/kaggle/working/finbert-ng",
        # A small labelled set overfits fast; 3 epochs at a low LR is enough to
        # adapt the head without washing out the pretrained financial knowledge.
        num_train_epochs=3,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=True,
        logging_steps=20,
        report_to="none",
    ),
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

## 4. Compare against the base model before shipping

Fine-tuning is only worth deploying if it beats the checkpoint already in
production, on the same held-out set.

In [ ]:
from transformers import pipeline

base = pipeline("text-classification", model=BASE_MODEL, device=0)
label_to_id = {"negative": 0, "neutral": 1, "positive": 2}

base_preds = [
    label_to_id[p["label"].lower()] for p in base(eval_df["headline"].tolist(), batch_size=32)
]
base_acc = (np.array(base_preds) == eval_df["label"].to_numpy()).mean()
tuned_acc = trainer.evaluate()["eval_accuracy"]

print(f"base  {base_acc:.3f}\ntuned {tuned_acc:.3f}")
assert tuned_acc > base_acc, "Fine-tune did not beat the base model — do not ship it."

## 5. Push to the Hub

Add your HF write token as a Kaggle secret named `HF_TOKEN`
(Add-ons → Secrets). Do not paste it into a cell — Kaggle notebooks committed
back to this repo are public.

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO_ID = "batestguy/finbert-ng-financial"
trainer.model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

print(f"Pushed. Now run:  gh variable set FINBERT_MODEL --body {REPO_ID}")